# Agent 4가지 핵심 역량 실습
## LangGraph & LangChain 으로 배우는 AI Agent

이 노트북은 Agentic AI 의 4가지 핵심 역량을 **실제 LLM 과 LangGraph** 를 사용해 단계별로 실습합니다.
기본 LLM 은 **로컬 Ollama + `qwen3:8b`** (무료·오프라인)이며, 클라우드 무료 API 인 **Google(Gemini)** 과
**NVIDIA build** 는 `.env` 의 `LLM_PROVIDER` 한 줄로 전환하는 비교 대상입니다(§0-B 참고).

| # | 역량 | 핵심 개념 | 실습 내용 |
|---|------|----------|---------|
| 1 | **Tool Use** | LLM 이 외부 도구를 호출 | @tool 데코레이터, bind_tools(), 자동 루프 |
| 2 | **Memory** | 대화 기록 유지 및 활용 | 메시지 히스토리, LangGraph MemorySaver |
| 3 | **Planning** | 복잡한 문제를 단계별 분해 | 구조화 출력(Pydantic), 계획 자동 실행 |
| 4 | **Reasoning** | 논리적 단계별 추론 | Chain of Thought, LangGraph ReAct Agent |
| 5 | **통합** | 4가지 역량 결합 | create_agent + MemorySaver |

> 📦 **환경 설치·실행 명령**은 [`env_guides/M01_2_core_capabilities.md`](env_guides/M01_2_core_capabilities.md) 에 별도 정리되어 있습니다.
> 반복·공통 코드는 [`agentic_lib/`](agentic_lib) 라이브러리(tools·memory·planning·react·bootstrap·capabilities)로 분리해 두었습니다.

### 전제 조건
- `M01_1_intro.ipynb` 완료 (uv 환경 구성, `.env` 파일 생성)
- 기본값: `notebooks/.env` 에 `LLM_PROVIDER=ollama` + Ollama 서버 실행(`ollama serve`) + 모델 다운로드(`ollama pull qwen3:8b`)
- (선택) 클라우드 비교: `LLM_PROVIDER=google` + `GOOGLE_API_KEY`, 또는 `LLM_PROVIDER=nvidia` + `NVIDIA_API_KEY`(build.nvidia.com)


---
## 0. 환경 설정


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(''))  # notebooks/ 를 import 경로에 추가

import utils
utils.reload_env()  # .env 재로드 (LLM_PROVIDER 등 갱신) + 현재 공급자 상태 출력

from utils import (
    uv_install, get_llm, test_llm_connection, LLM_PROVIDER,
    NVIDIA_API_KEY, NVIDIA_MODEL,   # NVIDIA build(무료 크레딧) 접속용 — §0-B
)

# 반복/공통 코드는 agentic_lib 라이브러리로 분리되어 있습니다(이 노트북 전반에서 사용).
from agentic_lib import bootstrap, tools, memory, planning, react
from agentic_lib.bootstrap import to_text          # 공급자 무관 응답 정규화(<think> 제거 포함)
from agentic_lib.capabilities import (             # Planning 심화: 구조화 출력 스키마 + 실행기
    PlanStep, ExecutionPlan, Status, execute_plan,
)

# LangGraph / LangChain 패키지 설치
uv_install([
    'langgraph>=0.2.0',
    'langchain>=0.3.0',
    'langchain-core>=0.3.0',
    'pydantic>=2.0.0',
])

llm = test_llm_connection()
print(f'\n준비 완료: {LLM_PROVIDER} 공급자 사용')


LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct


[uv] 설치 완료: ['langgraph>=0.2.0', 'langchain>=0.3.0', 'langchain-core>=0.3.0', 'pydantic>=2.0.0']


LLM 연결 성공 [nvidia]: 1+1은 2가 됩니다.

준비 완료: nvidia 공급자 사용


---
## 0-B. (선택) NVIDIA build 클라우드 모델 사용

로컬 Ollama 대신(또는 함께) **NVIDIA build**(build.nvidia.com, 무료 크레딧)의 모델을 써 볼 수 있습니다.
`utils.get_llm('nvidia')` 가 `ChatNVIDIA` 를 돌려주므로, 이 노트북의 나머지 코드와 **동일한 인터페이스**(`.invoke()` / `bind_tools()`)로 동작합니다.

| 항목 | 값 |
|---|---|
| 접속 | `utils.get_llm('nvidia')` (= `ChatNVIDIA`) |
| 키 | `.env` 의 `NVIDIA_API_KEY` (build.nvidia.com 에서 무료 발급) |
| 모델 | `.env` 의 `NVIDIA_MODEL` (기본 `meta/llama-3.1-8b-instruct`) |

- **개별 호출**: 아래 셀처럼 `nvidia_llm = utils.get_llm('nvidia')` 로 바로 사용.
- **노트북 전체를 NVIDIA 로**: `.env` 에서 `LLM_PROVIDER=nvidia` 로 바꾸고 §0 셀부터 다시 실행하면 `llm` 자체가 NVIDIA 모델이 됩니다(코드 수정 불필요).
- 키가 없으면 오류 없이 **발급 안내만** 출력합니다. 자세한 예제는 [`M02_0_free_llm_api.ipynb`](M02_0_free_llm_api.ipynb) 참고.

In [2]:
# === (선택) NVIDIA build 클라우드 모델 사용 데모 ===
# NVIDIA 커넥터 설치(이미 있으면 빠르게 통과)
uv_install(['langchain-nvidia-ai-endpoints'])

from langchain_core.messages import HumanMessage, SystemMessage

nvidia_llm = None
if not NVIDIA_API_KEY:
    print('NVIDIA_API_KEY 미설정 — https://build.nvidia.com 에서 무료 발급 후 .env 에 추가하세요.')
else:
    try:
        nvidia_llm = get_llm('nvidia')                 # ChatNVIDIA(model=NVIDIA_MODEL) 반환
        demo = nvidia_llm.invoke([
            SystemMessage(content='간결하게 한국어로 답하세요.'),
            HumanMessage(content='LangGraph 를 한 문장으로 설명해줘.'),
        ])
        print(f'[NVIDIA build · {NVIDIA_MODEL}]')
        print(' ', to_text(demo.content))              # 공급자 무관 응답 정규화
    except Exception as e:
        print(f'[NVIDIA 호출 실패] {type(e).__name__}: {e}')
        print('  → 모델명이 build.nvidia.com 에서 제공되는지, 키/크레딧이 유효한지 확인하세요.')

# 참고: 이 노트북의 Tool Use·에이전트 셀도 동일한 방식으로 nvidia_llm 을 쓸 수 있습니다
#      (도구 호출을 지원하는 NVIDIA 모델일 때). 예) llm_with_tools = nvidia_llm.bind_tools(tool_list)
#      전체를 NVIDIA 로 실행하려면 .env 의 LLM_PROVIDER=nvidia 로 바꾸고 §0 부터 다시 실행하세요(코드 수정 불필요).

[uv] 설치 완료: ['langchain-nvidia-ai-endpoints']


[NVIDIA build · meta/llama-3.1-8b-instruct]
  LangGraph는 언어 모델링을 위한 그래프 기반의 프레임워크로, 자연어 처리 및 언어 모델링을 위한 다양한 기능을 제공합니다.


---
## 1. Tool Use — LLM 이 외부 도구를 사용하는 방법

Tool Use 는 LLM 이 단순 텍스트 생성을 넘어 **외부 함수(도구)를 직접 호출**할 수 있는 능력입니다.

```
사용자 질문
    ↓
LLM 분석 → 어떤 도구를? 어떤 인자로?
    ↓
도구(함수) 실행
    ↓
실행 결과를 LLM 에 전달
    ↓
최종 자연어 답변 생성
```

**핵심 API**
- `@tool` 데코레이터: 일반 Python 함수를 LangChain 도구로 변환
- `llm.bind_tools(tools)`: LLM 에 도구 목록 전달 (스키마 자동 생성)
- `ToolMessage`: 도구 실행 결과를 메시지로 LLM 에 전달


### 1-1. 도구 정의 (`@tool` 데코레이터)


In [3]:
# 공통 도구는 agentic_lib.tools 에서 가져옵니다(calculator / get_current_time / search_web).
# (노트북마다 도구를 재정의하지 않고 라이브러리 한 곳에서 관리 — 중복 정의 제거)
tool_list = tools.BASIC_TOOLS               # [calculator, get_current_time, search_web]
tool_map = {t.name: t for t in tool_list}   # 이름 → 도구 매핑(아래 셀들에서 도구 실행에 사용)

print('등록된 도구 목록:')
for t in tool_list:
    print(f'  - {t.name}: {t.description}')


등록된 도구 목록:
  - calculator: 수학 수식을 계산합니다. 예: '2 ** 10', 'sqrt(144)', '(3+4)*5'.

    Python 수식 문법을 사용하며 sqrt, sin, log, pi 같은 math 함수/상수를 쓸 수 있습니다.
  - get_current_time: 현재 날짜와 시간을 'YYYY-MM-DD HH:MM:SS' 형식으로 반환합니다.

    인자가 없는 도구를 허용하지 않는 일부 공급자를 위해 사용하지 않는 dummy 인자를 둡니다.
  - search_web: 웹을 검색합니다. 기본은 DuckDuckGo 실제 검색이며, 오프라인/실패 시 시뮬레이션으로 폴백합니다.

    - DuckDuckGo(ddgs)는 API 키가 필요 없습니다.
    - 재현 가능한 결정론적 실행을 원하면 환경변수 `WEB_SEARCH_MODE=sim` 로 시뮬레이션을 강제하세요.
    - ddgs 미설치·네트워크 차단·레이트리밋 시에는 자동으로 시뮬레이션 결과로 폴백합니다.


### 1-2. LLM 에 도구 바인딩 — 단일 도구 호출 흐름


In [4]:
from langchain_core.messages import HumanMessage, ToolMessage

# 1) LLM 에 도구 목록 바인딩
llm_with_tools = llm.bind_tools(tool_list)

question = '2의 15제곱이 얼마야?'
print(f'[질문] {question}\n')

# 2) LLM 이 어떤 도구를 어떤 인자로 호출할지 결정
response = llm_with_tools.invoke([HumanMessage(content=question)])

print(f'[LLM 응답] tool_calls 수: {len(response.tool_calls)}개')
if response.tool_calls:
    tc = response.tool_calls[0]
    print(f"  → 선택 도구 : {tc['name']}")
    print(f"  → 도구 인자 : {tc['args']}")

    # 3) 도구 실행
    tool_result = tool_map[tc['name']].invoke(tc['args'])
    print(f'  → 도구 결과 : {tool_result}')

    # 4) 도구 결과를 LLM 에 전달 → 최종 자연어 답변
    final_messages = [
        HumanMessage(content=question),
        response,
        ToolMessage(content=str(tool_result), tool_call_id=tc['id'])
    ]
    final_resp = llm_with_tools.invoke(final_messages)
    print(f'\n[최종 답변] {to_text(final_resp.content)}')  # to_text 로 list/think 정규화


[질문] 2의 15제곱이 얼마야?



[LLM 응답] tool_calls 수: 1개
  → 선택 도구 : calculator
  → 도구 인자 : {'expression': '2 ** 15'}
  → 도구 결과 : 2 ** 15 = 32768



[최종 답변] 2의 15제곱은 32,768입니다.


### 1-3. 자동 Tool Call 루프 — 여러 도구를 순차 처리


In [5]:
from langchain_core.messages import AIMessage  # 병렬 tool_call 을 단일-호출 턴으로 분리해 기록

def run_tool_agent(question: str, max_steps: int = 6) -> str:
    """질문을 받아 도구 자동 선택·실행을 반복하다가 최종 답변을 돌려준다.

    한 응답에 여러 tool_call 이 담겨도, 각 호출을 '단일 tool_call 을 가진 별도 AIMessage +
    그에 대응하는 ToolMessage' 쌍으로 히스토리에 기록한다. 이렇게 하면
    NVIDIA build 의 llama-3.1 처럼 '한 번에 하나의 tool_call 만 지원'하는 모델에서도
    (`This model only supports single tool-calls at once!`) 오류 없이 동작한다.
    """
    print(f'[질문] {question}')
    print('-' * 55)

    messages = [HumanMessage(content=question)]

    for step in range(1, max_steps + 1):
        response = llm_with_tools.invoke(messages)

        # 도구 호출 없음 → 최종 답변
        if not response.tool_calls:
            messages.append(response)
            answer = to_text(response.content)  # 공급자 무관 정규화
            print(f'[최종 답변] {answer}')
            return answer

        # 도구 호출 처리 — 각 tool_call 을 단일-호출 턴으로 나눠 기록(단일호출만 지원 모델 호환)
        for tc in response.tool_calls:
            print(f"[Step {step}] {tc['name']}({tc['args']})")
            messages.append(AIMessage(content='', tool_calls=[tc]))
            result = tool_map[tc['name']].invoke(tc['args'])
            print(f'         → {result}')
            messages.append(ToolMessage(content=str(result), tool_call_id=tc['id']))

    return '최대 단계 초과'


print('=' * 60)
print('테스트 1: 여러 도구 동시 사용')
print('=' * 60)
run_tool_agent('지금 몇 시야? 그리고 3의 7제곱도 계산해줘.')

print()
print('=' * 60)
print('테스트 2: 검색 도구 사용')
print('=' * 60)
run_tool_agent('LangGraph 에 대해 검색해줘.')

테스트 1: 여러 도구 동시 사용
[질문] 지금 몇 시야? 그리고 3의 7제곱도 계산해줘.
-------------------------------------------------------


[Step 1] get_current_time({'dummy': ''})
         → 2026-07-25 15:50:39
[Step 1] calculator({'expression': '3 ** 7'})
         → 3 ** 7 = 2187


[최종 답변] The current time is 2026-07-25 15:50:39 and 3 to the power of 7 is 2187.

테스트 2: 검색 도구 사용
[질문] LangGraph 에 대해 검색해줘.
-------------------------------------------------------


[Step 1] search_web({'query': 'LangGraph'})


         → 'LangGraph' 웹 검색 결과 상위 5건:
1. Laographer
   Folklore is the body of expressive culture shared by a particular group of people, culture or subculture. This includes oral traditions such as tales, legends, 
   https://en.wikipedia.org/wiki/Laographer
2. LangGraph
   LangGraph is an MIT-licensed open-source low-level orchestration framework developed by the LangChain team for building resilient, stateful, multi-actor AI agen
   https://grokipedia.com/page/LangGraph
3. LangGraph: Agent Orchestration Framework for Reliable AI Agents
   Build controllable agents with LangGraph, our low-level agent orchestration framework
   https://www.langchain.com/langgraph
4. GitHub - langchain-ai/langgraph: Build resilient agents. · GitHub
   May 6, 2026 - Trusted by companies shaping the future of agents – including Klarna, Replit, Elastic, and more – LangGraph is a low-level orchestration framework
   https://github.com/langchain-ai/langgraph
5. What is LangGraph? | IBM
   2 weeks ago - Lang

[Step 2] search_web({'query': 'LangGraph'})


         → 'LangGraph' 웹 검색 결과 상위 5건:
1. Laographer
   Folklore is the body of expressive culture shared by a particular group of people, culture or subculture. This includes oral traditions such as tales, legends, 
   https://en.wikipedia.org/wiki/Laographer
2. LangGraph
   LangGraph is an MIT-licensed open-source low-level orchestration framework developed by the LangChain team for building resilient, stateful, multi-actor AI agen
   https://grokipedia.com/page/LangGraph
3. LangGraph: Agent Orchestration Framework for Reliable AI Agents
   Build controllable agents with LangGraph, our low-level agent orchestration framework
   https://www.langchain.com/langgraph
4. GitHub - langchain-ai/langgraph: Build resilient agents. · GitHub
   May 6, 2026 - Trusted by companies shaping the future of agents – including Klarna, Replit, Elastic, and more – LangGraph is a low-level orchestration framework
   https://github.com/langchain-ai/langgraph
5. What is LangGraph? | IBM
   2 weeks ago - Lang

[최종 답변] The search results for "LangGraph" are as follows:

1. Laographer
   Folklore is the body of expressive culture shared by a particular group of people, culture or subculture. This includes oral traditions such as tales, legends, 
   https://en.wikipedia.org/wiki/Laographer

2. LangGraph
   LangGraph is an MIT-licensed open-source low-level orchestration framework developed by the LangChain team for building resilient, stateful, multi-actor AI agen
   https://grokipedia.com/page/LangGraph

3. LangGraph: Agent Orchestration Framework for Reliable AI Agents
   Build controllable agents with LangGraph, our low-level agent orchestration framework
   https://www.langchain.com/langgraph

4. GitHub - langchain-ai/langgraph: Build resilient agents. · GitHub
   May 6, 2026 - Trusted by companies shaping the future of agents – including Klarna, Replit, Elastic, and more – LangGraph is a low-level orchestration framework
   https://github.com/langchain-ai/langgraph

5. What is LangGraph? |

'The search results for "LangGraph" are as follows:\n\n1. Laographer\n   Folklore is the body of expressive culture shared by a particular group of people, culture or subculture. This includes oral traditions such as tales, legends, \n   https://en.wikipedia.org/wiki/Laographer\n\n2. LangGraph\n   LangGraph is an MIT-licensed open-source low-level orchestration framework developed by the LangChain team for building resilient, stateful, multi-actor AI agen\n   https://grokipedia.com/page/LangGraph\n\n3. LangGraph: Agent Orchestration Framework for Reliable AI Agents\n   Build controllable agents with LangGraph, our low-level agent orchestration framework\n   https://www.langchain.com/langgraph\n\n4. GitHub - langchain-ai/langgraph: Build resilient agents. · GitHub\n   May 6, 2026 - Trusted by companies shaping the future of agents – including Klarna, Replit, Elastic, and more – LangGraph is a low-level orchestration framework\n   https://github.com/langchain-ai/langgraph\n\n5. What is L

---
## 2. Memory — 기억하는 에이전트

AI Agent 의 기억은 두 층위로 나뉩니다.

| 종류 | 범위 | 구현 방식 |
|------|------|----------|
| **단기 기억** | 현재 대화 세션 | 메시지 히스토리 (Python 리스트) |
| **장기 기억** | 세션 간 지속 | LangGraph `MemorySaver` / 외부 DB |

LangGraph 는 **체크포인터(Checkpointer)** 를 통해 그래프 실행 상태를 자동으로 저장하고
`thread_id` 로 사용자/세션을 구분합니다.


### 2-1. 단기 기억 — 메시지 히스토리


In [6]:
from langchain_core.messages import SystemMessage, HumanMessage

print('=' * 60)
print('단기 기억: 메시지 리스트로 대화 맥락 유지')
print('=' * 60)

conversation = [
    SystemMessage(content='당신은 사용자 정보를 꼼꼼히 기억하는 AI 어시스턴트입니다.')
]

def chat(user_input: str) -> str:
    """대화 리스트에 사용자 입력을 추가하고 LLM 응답을 받아 누적한다."""
    conversation.append(HumanMessage(content=user_input))
    reply = llm.invoke(conversation)
    conversation.append(reply)
    return to_text(reply.content)  # 출력용으로 정규화한 문자열 반환


turns = [
    '안녕하세요! 저는 박지수이고 부산에 살아요.',
    '저는 데이터 사이언티스트로 LangGraph 를 공부하고 있어요.',
    '제가 어디 살고 무슨 일을 한다고 했었죠?',
]

for i, user_msg in enumerate(turns, 1):
    print(f'\n[대화 {i}]')
    print(f'  사용자: {user_msg}')
    reply = chat(user_msg)
    print(f'  에이전트: {reply[:120]}')

print(f'\n누적 메시지 수: {len(conversation)}개')


단기 기억: 메시지 리스트로 대화 맥락 유지

[대화 1]
  사용자: 안녕하세요! 저는 박지수이고 부산에 살아요.


  에이전트: 안녕하세요 박지수 씨! 부산에 살고 계신다는 것을 알게 되었습니다. 부산은 좋은 도시라고 들었는데요. 어떤 활동이나 취미가 있으신가요?

[대화 2]
  사용자: 저는 데이터 사이언티스트로 LangGraph 를 공부하고 있어요.


  에이전트: 데이터 사이언티스트로 LangGraph를 공부하고 계신다는 것을 알게 되었습니다. LangGraph는 자연어 처리와 언어 모델링에 사용되는 그래프 기반의 언어 모델링 기술입니다. LangGraph를 공부하시는 것은 

[대화 3]
  사용자: 제가 어디 살고 무슨 일을 한다고 했었죠?


  에이전트: 박지수 씨라고 하셨고, 부산에 살고 계신다는 것을 기억했습니다. 또한 데이터 사이언티스트로 LangGraph를 공부하고 계신다는 것을 알게 되었습니다.

누적 메시지 수: 7개


### 2-2. 장기 기억 — LangGraph MemorySaver

`MemorySaver` 는 그래프의 상태(메시지 전체)를 `thread_id` 단위로 **메모리에 영구 저장**합니다.
프로세스가 살아있는 한 세션 간에도 기억이 유지됩니다.  
(프로덕션에서는 `SqliteSaver` / `PostgresSaver` 로 교체하여 영구 저장 가능)


In [7]:
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import MemorySaver

# ── 그래프 정의 ──────────────────────────────────────────────
def chatbot_node(state: MessagesState) -> dict:
    """시스템 프롬프트 + 누적 메시지로 LLM 을 호출하는 단일 노드."""
    system = SystemMessage(content='당신은 사용자 정보를 기억하는 AI 어시스턴트입니다.')
    response = llm.invoke([system] + state['messages'])
    return {'messages': [response]}

builder = StateGraph(MessagesState)
builder.add_node('chatbot', chatbot_node)
builder.add_edge(START, 'chatbot')

memory_saver = MemorySaver()
app = builder.compile(checkpointer=memory_saver)

# ── 유틸 함수 ────────────────────────────────────────────────
def chat_with_thread(thread_id: str, message: str) -> str:
    """thread_id 단위로 상태를 유지하며 한 턴 대화한다."""
    config = {'configurable': {'thread_id': thread_id}}
    result = app.invoke({'messages': [HumanMessage(content=message)]}, config=config)
    return to_text(result['messages'][-1].content)  # 정규화 문자열


print('=' * 60)
print('LangGraph MemorySaver: thread_id 별 독립 기억')
print('=' * 60)

# Thread 'alice': 앨리스의 대화
print('\n[Thread: alice]')
r = chat_with_thread('alice', '안녕! 나는 앨리스야. 뉴욕에 살고 있어.')
print(f'  1번째: {r[:80]}')

r = chat_with_thread('alice', 'AI 에이전트를 공부하고 있어.')
print(f'  2번째: {r[:80]}')

r = chat_with_thread('alice', '내가 어디 살고 뭘 공부한다고 했지?')
print(f'  3번째: {r[:120]}')

# Thread 'bob': 밥의 대화 (앨리스와 완전히 독립)
print('\n[Thread: bob — 앨리스 정보 없음]')
r = chat_with_thread('bob', '내가 어디 살고 뭘 공부한다고 했지?')
print(f'  답변: {r[:100]}')


LangGraph MemorySaver: thread_id 별 독립 기억

[Thread: alice]


  1번째: 안녕하세요 앨리스! 뉴욕에 살고 계신가요? 뉴욕은 정말 멋진 도시입니다. 어떤 활동이나 관심사를 가지고 계신가요?


  2번째: AI 에이전트를 공부하시는군요. AI는 정말 흥미로운 분야입니다. 뉴욕에 살면서 AI를 공부하시는 것은 정말 좋은 기회인 것 같습니다. 어떤 부


  3번째: 앨리스님께서 뉴욕에 살고 AI 에이전트를 공부한다고 말씀하셨습니다.

[Thread: bob — 앨리스 정보 없음]


  답변: 당신이 이전에 말한 정보를 기억하고 있지 않습니다. 이전에 어떤 정보를 말한 적이 없기 때문에 기억할 수 없습니다. 하지만 지금부터는 당신과 대화할 때 기억할 수 있습니다.


### 2-3. Thread 별 메모리 상태 확인


In [8]:
from langchain_core.messages import AIMessage

print('=' * 60)
print('저장된 대화 히스토리 확인')
print('=' * 60)

for thread_id in ['alice', 'bob']:
    config = {'configurable': {'thread_id': thread_id}}
    state = app.get_state(config)
    msgs = state.values.get('messages', [])
    print(f'\n[{thread_id}] 메시지 수: {len(msgs)}개')
    for msg in msgs:
        role = '사용자' if isinstance(msg, HumanMessage) else '에이전트'
        print(f'  [{role}] {to_text(msg.content)[:70]}')  # content 정규화 후 출력


저장된 대화 히스토리 확인

[alice] 메시지 수: 6개
  [사용자] 안녕! 나는 앨리스야. 뉴욕에 살고 있어.
  [에이전트] 안녕하세요 앨리스! 뉴욕에 살고 계신가요? 뉴욕은 정말 멋진 도시입니다. 어떤 활동이나 관심사를 가지고 계신가요?
  [사용자] AI 에이전트를 공부하고 있어.
  [에이전트] AI 에이전트를 공부하시는군요. AI는 정말 흥미로운 분야입니다. 뉴욕에 살면서 AI를 공부하시는 것은 정말 좋은 기회인 것 
  [사용자] 내가 어디 살고 뭘 공부한다고 했지?
  [에이전트] 앨리스님께서 뉴욕에 살고 AI 에이전트를 공부한다고 말씀하셨습니다.

[bob] 메시지 수: 2개
  [사용자] 내가 어디 살고 뭘 공부한다고 했지?
  [에이전트] 당신이 이전에 말한 정보를 기억하고 있지 않습니다. 이전에 어떤 정보를 말한 적이 없기 때문에 기억할 수 없습니다. 하지만 지


---
## 3. Planning — 계획하는 에이전트

Planning 은 복잡한 목표를 **실행 가능한 단계들로 분해**하는 능력입니다.

LangChain 의 **구조화 출력(Structured Output)** 을 사용하면
LLM 이 자유 텍스트 대신 **Pydantic 스키마에 맞는 JSON** 으로 계획을 반환합니다.

```
목표 입력
    ↓
LLM (with_structured_output) → Pydantic 객체
    ↓
단계별 자동 실행 (도구 / LLM)
    ↓
최종 결과 취합
```


### 3-1. 구조화 출력으로 실행 계획 생성


In [9]:
# Planning 심화 구현(PlanStep / ExecutionPlan / Status / execute_plan)은
# agentic_lib.capabilities 로 분리되어 있습니다(셀 상단에서 import 완료).

# LLM 에 구조화 출력 스키마 적용 — 자유 텍스트 대신 ExecutionPlan(JSON) 으로 응답
planner = llm.with_structured_output(ExecutionPlan)

# ── 현재 시각 grounding ───────────────────────────────────────────────────
# LLM 은 학습 시점(예: llama-3.1 은 2023년 무렵) 이후의 '오늘 날짜'를 모른다.
# 그래서 시간 관련 필드(estimated_duration 등)를 물으면 2023 같은 과거 날짜를 지어낸다.
# 실제 현재 시각을 get_current_time 도구로 구해 프롬프트에 주입하면 '현재 시점'을 기준으로
# 답하게 되어, 스테일한 2023 날짜 대신 올바른 값이 나온다.
now = tool_map['get_current_time'].invoke({'dummy': ''})   # 예: '2026-07-11 11:23:02'

plan_request = (
    f'현재 시각: {now}\n'
    '목표: AI 에이전트 기술 동향 분석 보고서 작성\n'
    '사용 가능한 도구: calculator (수학 계산), get_current_time (현재 시간), search_web (웹 검색)\n'
    '각 단계에서 적절한 도구를 활용하는 5단계 계획을 작성하세요.\n'
    'estimated_duration 은 날짜가 아니라 위 현재 시각 기준의 "예상 소요 시간"(예: "약 3시간", "2일")으로 채우세요.'
)

plan: ExecutionPlan = planner.invoke(plan_request)

print(f'[기준 현재 시각] {now}')            # 프롬프트에 주입한 실제 현재 시각
print(f'[목표] {plan.goal}')
print(f'[예상 소요 시간] {plan.estimated_duration}')
print(f'\n[실행 계획] — {len(plan.steps)}단계:')
for step in plan.steps:
    dep = f'  (의존: Step {step.depends_on})' if step.depends_on else ''
    tool_tag = f'  [도구: {step.tool}]' if step.tool else ''
    print(f'  Step {step.step_num}: {step.action}{tool_tag}{dep}')
    print(f'    예상 결과: {step.expected_output}')

[기준 현재 시각] 2026-07-25 15:50:53
[목표] AI 에이전트 기술 동향 분석 보고서 작성
[예상 소요 시간] 약 5일

[실행 계획] — 5단계:
  Step 1: 과제 정의 및 목표 설정
    예상 결과: 과제 정의서 및 목표 설정서 작성
  Step 2: 현재 기술 동향 분석
    예상 결과: AI 에이전트 기술 동향 분석 보고서 초안 작성
  Step 3: 웹 검색 및 데이터 수집
    예상 결과: AI 에이전트 기술 동향 분석 보고서 주요 데이터 수집
  Step 4: 보고서 작성 및 편집
    예상 결과: AI 에이전트 기술 동향 분석 보고서 최종본 작성
  Step 5: 보고서 검토 및 최종 제출
    예상 결과: AI 에이전트 기술 동향 분석 보고서 최종 제출


### 3-2. 계획 자동 실행


In [10]:
# execute_plan 은 agentic_lib.capabilities 로 분리(의존성 순서대로 도구/LLM 실행).
# tool_map 과 llm 을 넘겨 호출하면 단계별 진행 상황을 출력하며 결과 딕셔너리를 돌려준다.
print('=' * 60)
print('계획 자동 실행')
print('=' * 60)
results = execute_plan(plan, tool_map, llm)


계획 자동 실행
[목표] AI 에이전트 기술 동향 분석 보고서 작성

[실행중] Step 1: 과제 정의 및 목표 설정


  → 과제 정의 및 목표 설정은 프로젝트를 성공적으로 완료하기 위한 첫 번째 단계입니다. 과제 정의 및 목표 설정을 통해 프로젝트의 목적, 범위, 시간표, 자원, 그리고 성공 기준을 명

[실행중] Step 2: 현재 기술 동향 분석


  → 현재 기술 동향은 빠르게 변화하고 있습니다. 하지만 몇 가지 주요한 기술 동향을 살펴보겠습니다.

1. **인공지능 (AI) 및 머신 러닝 (ML)**: 인공지능과 머신 러닝은 최

[실행중] Step 3: 웹 검색 및 데이터 수집


  → 웹 검색 및 데이터 수집은 웹에서 데이터를 수집하고 분석하는 과정입니다. 이 과정을 통해 웹에서 다양한 데이터를 수집하고 분석할 수 있습니다. 웹 검색 및 데이터 수집의 예를 들어

[실행중] Step 4: 보고서 작성 및 편집


  → 보고서 작성 및 편집은 중요한 업무입니다. 보고서 작성은 정보를 정리하고 분석하여 보고서를 작성하는 과정을 의미하며, 보고서 편집은 작성된 보고서를 검토하고 수정하여 최종 버전을 

[실행중] Step 5: 보고서 검토 및 최종 제출


  → 보고서 검토 및 최종 제출을 위한 단계는 다음과 같습니다.

1. **보고서 검토**: 
 - 보고서의 내용을 확인하여 정확성과 완전성을 확인합니다.
 - 보고서의 구조와 형식이 

[완료] 5/5 단계 실행


---
## 4. Reasoning — 추론하는 에이전트

Reasoning 은 주어진 정보를 **논리적으로 분석하고 단계별로 사고**하는 능력입니다.

| 기법 | 설명 | 효과 |
|------|------|------|
| **Chain of Thought (CoT)** | "단계별로 생각하세요" 프롬프트 | 복잡한 수학·논리 문제 정확도 향상 |
| **ReAct** | Reasoning + Acting 결합 루프 | 도구 사용과 추론을 반복하며 문제 해결 |
| **Self-Reflection** | 자신의 답변을 재검토 | 오류 감소, 품질 향상 |


### 4-1. Chain of Thought — 일반 답변 vs 단계별 추론 비교


In [11]:
from langchain_core.prompts import ChatPromptTemplate

problem = (
    '학교 도서관에 책이 1,500권 있습니다.\n'
    '- 월요일: 전체의 20% 를 학생들이 빌렸습니다.\n'
    '- 화요일: 남은 책의 25% 가 추가로 빌려졌습니다.\n'
    '- 수요일: 빌린 책의 절반이 반납되었습니다.\n\n'
    '수요일 이후 도서관에 있는 책은 총 몇 권인가요?'
)

# 방법 A: 직접 답변 (추론 없음)
print('=' * 60)
print('방법 A: 직접 답변 (추론 과정 없음)')
print('=' * 60)
direct = llm.invoke([HumanMessage(content=problem)])
print(to_text(direct.content)[:300])  # 정규화 후 출력

# 방법 B: Chain of Thought
print('\n' + '=' * 60)
print('방법 B: Chain of Thought — 단계별 추론')
print('=' * 60)

cot_prompt = ChatPromptTemplate.from_messages([
    ('system',
     '당신은 정밀한 수학 문제 해결사입니다.\n'
     '반드시 다음 형식으로 단계별로 추론하세요:\n\n'
     '**단계 1:** [첫 번째 계산 — 수식과 결과 명시]\n'
     '**단계 2:** [두 번째 계산]\n'
     '...\n'
     '**검증:** [풀이 과정 간단 검토]\n'
     '**최종 답:** [숫자 결론]'),
    ('human', '{problem}')          # {problem} 은 나중에 채워질 자리(placeholder)
])

# ── cot_chain = cot_prompt | llm ──────────────────────────────────────────
# LCEL(LangChain Expression Language) 문법. '|'(파이프)로 프롬프트→LLM 을 하나의
# 실행 체인으로 연결한다. '|' 는 왼쪽의 출력을 오른쪽의 입력으로 흘려보낸다(셸 파이프와 같은 발상).
#   {'problem': ...} ─▶ cot_prompt(템플릿 채움) ─▶ 완성된 메시지 ─▶ llm(호출) ─▶ AIMessage
# 즉 cot_chain 은 '입력 dict → 프롬프트 완성 → LLM 호출 → 응답'을 하는 하나의 합성 Runnable 이다.
# (ChatPromptTemplate 과 llm 둘 다 Runnable 이라 파이프로 이을 수 있고, 결과도 다시 Runnable)
cot_chain = cot_prompt | llm

# .invoke() 한 번이 내부적으로 (1) 프롬프트 채우기 → (2) LLM 호출 두 단계를 자동 실행한다.
# 파이프 없이 풀어 쓰면 아래 두 줄과 동일:
#     messages = cot_prompt.invoke({'problem': problem})   # 1단계: 프롬프트 완성
#     cot_result = llm.invoke(messages)                    # 2단계: LLM 호출
# 이렇게 체인으로 만들어 두면 문제만 바꿔 재사용/확장(예: | StrOutputParser())하기 쉽다.
cot_result = cot_chain.invoke({'problem': problem})
print(to_text(cot_result.content))  # 정규화 후 출력

방법 A: 직접 답변 (추론 과정 없음)


월요일에 학생들이 20%의 책을 빌렸다면, 20%는 0.2로 계산됩니다. 

1,500 * 0.2 = 300권

월요일에 학생들이 빌린 책의 수는 300권입니다. 

도서관에 남은 책의 수는 1,500 - 300 = 1,200권입니다.

화요일에 남은 책의 25%가 추가로 빌려졌습니다. 

1,200 * 0.25 = 300권

화요일에 추가로 빌린 책의 수는 300권입니다. 

도서관에 남은 책의 수는 1,200 - 300 = 900권입니다.

수요일에 빌린 책의 절반, 즉 300/2 = 150권이 반납되었습니다.

도서관에 남은 

방법 B: Chain of Thought — 단계별 추론


**단계 1:** 월요일에 학생들이 빌린 책의 수를 계산합니다. 
월요일에 학생들이 빌린 책의 수 = 전체 책의 20% = 1,500 * 0.2 = 300권

**단계 2:** 월요일에 학생들이 빌린 책을 제외한 남은 책의 수를 계산합니다.
남은 책의 수 = 전체 책의 수 - 월요일에 학생들이 빌린 책의 수 = 1,500 - 300 = 1,200권

**단계 3:** 화요일에 추가로 빌린 책의 수를 계산합니다.
화요일에 추가로 빌린 책의 수 = 남은 책의 수 * 25% = 1,200 * 0.25 = 300권

**단계 4:** 화요일에 추가로 빌린 책을 포함한 현재 빌린 책의 수를 계산합니다.
현재 빌린 책의 수 = 월요일에 학생들이 빌린 책의 수 + 화요일에 추가로 빌린 책의 수 = 300 + 300 = 600권

**단계 5:** 수요일에 반납된 책의 수를 계산합니다.
수요일에 반납된 책의 수 = 현재 빌린 책의 수 * 50% = 600 * 0.5 = 300권

**단계 6:** 수요일에 반납된 책을 제외한 현재 빌린 책의 수를 계산합니다.
현재 빌린 책의 수 = 현재 빌린 책의 수 - 수요일에 반납된 책의 수 = 600 - 300 = 300권

**단계 7:** 수요일 이후 도서관에 있는 책의 수를 계산합니다.
수요일 이후 도서관에 있는 책의 수 = 전체 책의 수 - 현재 빌린 책의 수 = 1,500 - 300 = 1,200권

**검증:** 계산 과정을 확인하여 오류가 없는지 확인합니다.

**최종 답:** 1,200


### 4-2. ReAct 패턴 — LangGraph 내장 에이전트

`create_agent` 는 **Reasoning + Acting** 루프를 LangGraph 그래프로 구현합니다.

```
[Thought] 무엇을 해야 하는가? 어떤 도구가 필요한가?
    ↓
[Action] 도구 호출
    ↓
[Observation] 도구 실행 결과 확인
    ↓
(반복 또는 종료)
    ↓
[Final Answer] 최종 자연어 답변
```


In [12]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# ── 에이전트 전용 모델 교체 (중요) ─────────────────────────────────────────
# ReAct·통합 에이전트는 '여러 단계에 걸친 도구 루프(Thought→Action→Observation 반복)'라서
# 도구 호출(tool calling)이 강한 모델이 필요하다. 그런데 이 노트북의 기본 llm 은
# NVIDIA build 의 meta/llama-3.1-8b-instruct 이고, 이 모델은 에이전트 용도로 두 한계가 있다:
#   (1) 멀티 콜 미지원 — '한 번에 하나의 tool_call 만' 허용한다.
#       (여러 tool_call 을 만들면: This model only supports single tool-calls at once! [500] 오류)
#   (2) 에이전트 루프가 약함 — 실제로 기본 llm 으로 돌리면:
#       · 테스트1: 도구를 실행하지 않고 tool_call JSON 을 최종 답변 '텍스트'로 그대로 뱉는다
#                 (예: {"name": "calculator", "parameters": {"expression": "11:53:01 % 7"}}).
#       · 테스트2: 요약을 못 하고 search_web 를 반복 호출해 결과 목록의 '1.'번이 계속 반복 출력된다.
# 그래서 이 두 셀(ReAct·통합)에서는 도구 호출이 안정적인 NVIDIA build 대형 모델로 교체해 실행한다.
# 참고: 원래 쓰려던 qwen/qwen3-next-80b-a3b-thinking 은 2026-05-21 EOL(410 Gone)이라 사용 불가 →
#       살아있는 동일 계열 instruct 버전(qwen/qwen3-next-80b-a3b-instruct)을 대신 사용한다.
AGENT_MODEL = 'qwen/qwen3-next-80b-a3b-instruct'   # 멀티 스텝 도구 루프에 적합한 대형 모델
# timeout: 이 대형 모델은 멀티 스텝 도구 루프에서 한 번의 응답이 길어져,
#          기본 read timeout(60초)을 넘겨 ReadTimeout 이 난다. 넉넉히 5분으로 늘린다.
agent_llm = ChatNVIDIA(model=AGENT_MODEL, api_key=NVIDIA_API_KEY, temperature=0, timeout=300)
print(f'[에이전트 모델] {AGENT_MODEL} (기본 llm=llama-3.1-8b-instruct 대신 사용)')

react_memory = MemorySaver()

react_agent = create_agent(
    model=agent_llm,               # 기본 llm 이 아니라 도구 호출이 강한 agent_llm 사용
    tools=tool_list,
    checkpointer=react_memory,
    system_prompt=(
        '당신은 도구를 활용해 문제를 단계별로 해결하는 AI 에이전트입니다. '
        '필요한 도구를 적극 사용하고 한국어로 답변하세요.'
    ),
)


def run_react_agent(question: str, thread_id: str = 'react-1') -> str:
    """ReAct 에이전트를 실행하고 중간 도구 호출·결과·최종 답변을 출력한다."""
    print(f'[질문] {question}')
    print('-' * 55)

    config = {'configurable': {'thread_id': thread_id}}
    result = react_agent.invoke(
        {'messages': [HumanMessage(content=question)]},
        config=config
    )

    # 중간 과정(도구 호출) 표시
    from langchain_core.messages import AIMessage, ToolMessage as TM
    for msg in result['messages']:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  [ReAct → 도구 선택] {tc['name']}({tc['args']})")
        elif isinstance(msg, TM):
            print(f'  [ReAct → 도구 결과] {to_text(msg.content)[:80]}')

    final = to_text(result['messages'][-1].content)  # 정규화 후 출력
    print(f'\n[최종 답변] {final}')
    return final


print('=' * 60)
print('ReAct 테스트 1: 도구 추론 + 계산')
print('=' * 60)
run_react_agent(
    '지금 몇 시야? 그 시각의 분(minute)을 7로 나눈 나머지를 계산해줘.',
    thread_id='react-test-1'
)

print('\n' + '=' * 60)
print('ReAct 테스트 2: 검색 + 요약')
print('=' * 60)
run_react_agent(
    'AI 트렌드에 대해 검색하고 핵심을 3줄로 요약해줘.',
    thread_id='react-test-2'
)

[에이전트 모델] qwen/qwen3-next-80b-a3b-instruct (기본 llm=llama-3.1-8b-instruct 대신 사용)
ReAct 테스트 1: 도구 추론 + 계산
[질문] 지금 몇 시야? 그 시각의 분(minute)을 7로 나눈 나머지를 계산해줘.
-------------------------------------------------------


  [ReAct → 도구 선택] get_current_time({})
  [ReAct → 도구 결과] 2026-07-25 15:52:02
  [ReAct → 도구 선택] calculator({'expression': '52 % 7'})
  [ReAct → 도구 결과] 52 % 7 = 3

[최종 답변] 현재 시각은 2026년 7월 25일 15시 52분입니다.  
분(minute)인 52를 7로 나눈 나머지는 3입니다.

ReAct 테스트 2: 검색 + 요약
[질문] AI 트렌드에 대해 검색하고 핵심을 3줄로 요약해줘.
-------------------------------------------------------


  [ReAct → 도구 선택] search_web({'query': '최신 AI 트렌드 2024 핵심 동향'})
  [ReAct → 도구 결과] '최신 AI 트렌드 2024 핵심 동향' 웹 검색 결과 상위 5건:
1. 미래를 여는 열쇠 AGI: 관련 산업과 핵심 기업들의 역할
   최신 
  [ReAct → 도구 선택] search_web({'query': '2024년 AI 트렌드 멀티모델 오케스트레이션 자율 에이전트'})
  [ReAct → 도구 결과] '2024년 AI 트렌드 멀티모델 오케스트레이션 자율 에이전트' 웹 검색 결과 상위 5건:
1. AI 에이전트 오케스트레이션 완벽 가이드 202
  [ReAct → 도구 선택] search_web({'query': '2024년 AI 트렌드 신뢰성과 윤리적 AI 실용화'})
  [ReAct → 도구 결과] '2024년 AI 트렌드 신뢰성과 윤리적 AI 실용화' 웹 검색 결과 상위 5건:
1. 2024년 예상되는 AI 기술 트렌드 12가지 - 디지털

[최종 답변] 2024년 AI 트렌드의 핵심은 다음과 같습니다:

1. **멀티모델 오케스트레이션**이 주류로 부상: GPT-5, Claude, Gemini 등 다양한 AI 모델을 협업시켜 복잡한 작업을 자동으로 분배하고 실행하는 아키텍처가 기업과 연구기관에서 실용화되고 있습니다.

2. **자율 에이전트의 확산**: 단순한 응답을 넘어, 목표를 설정하고 도구를 선택하며 환경과 상호작용하는 자율 AI 에이전트가 마케팅, 개발, 고객 서비스 등 다양한 산업에 적용되고 있습니다.

3. **신뢰성과 윤리적 AI 강화**: 투명성, 편향 제거, 데이터 프라이버시를 고려한 윤리적 AI 개발이 기술적 진보와 병행되며, 규제와 표준화가 전 세계적으로 가속화되고 있습니다.


'2024년 AI 트렌드의 핵심은 다음과 같습니다:\n\n1. **멀티모델 오케스트레이션**이 주류로 부상: GPT-5, Claude, Gemini 등 다양한 AI 모델을 협업시켜 복잡한 작업을 자동으로 분배하고 실행하는 아키텍처가 기업과 연구기관에서 실용화되고 있습니다.\n\n2. **자율 에이전트의 확산**: 단순한 응답을 넘어, 목표를 설정하고 도구를 선택하며 환경과 상호작용하는 자율 AI 에이전트가 마케팅, 개발, 고객 서비스 등 다양한 산업에 적용되고 있습니다.\n\n3. **신뢰성과 윤리적 AI 강화**: 투명성, 편향 제거, 데이터 프라이버시를 고려한 윤리적 AI 개발이 기술적 진보와 병행되며, 규제와 표준화가 전 세계적으로 가속화되고 있습니다.'

---
## 5. 통합 에이전트 — 4가지 역량이 결합된 완성형 Agent

지금까지 배운 4가지 역량을 하나의 에이전트에 결합합니다.

| 역량 | 구현 방식 |
|------|----------|
| Tool Use | `bind_tools` / `create_agent` |
| Memory | `MemorySaver` + `thread_id` |
| Planning | LLM 의 내재적 계획 수립 능력 |
| Reasoning | ReAct 루프 (Thought → Action → Observation) |


In [13]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# ── 에이전트 전용 모델 교체 (중요) ─────────────────────────────────────────
# 통합 에이전트도 §4-2 ReAct 와 같은 멀티 스텝 도구 루프라, 기본 llm 인
# NVIDIA build 의 meta/llama-3.1-8b-instruct 로는 정상 동작하지 않는다:
#   (1) 멀티 콜 미지원 — '한 번에 하나의 tool_call 만' 허용
#       (This model only supports single tool-calls at once! [500] 오류).
#   (2) 에이전트 루프가 약함 — 기억(Memory) 질문에도 엉뚱하게 search_web 를 반복 호출하고,
#       저장한 사용자 정보를 제대로 회상하지 못한다(예: 이름을 '네이버의 …' 로 왜곡).
# 그래서 이 셀도 도구 호출이 안정적인 NVIDIA build 대형 모델로 교체해 실행한다.
# 참고: qwen/qwen3-next-80b-a3b-thinking 은 2026-05-21 EOL(410 Gone)이라, 살아있는 동일 계열
#       instruct 버전(qwen/qwen3-next-80b-a3b-instruct)을 사용한다.
AGENT_MODEL = 'qwen/qwen3-next-80b-a3b-instruct'   # 멀티 스텝 도구 루프에 적합한 대형 모델
# timeout: 이 대형 모델은 멀티 스텝 도구 루프에서 한 번의 응답이 길어져,
#          기본 read timeout(60초)을 넘겨 ReadTimeout 이 난다. 넉넉히 5분으로 늘린다.
agent_llm = ChatNVIDIA(model=AGENT_MODEL, api_key=NVIDIA_API_KEY, temperature=0, timeout=300)
print(f'[에이전트 모델] {AGENT_MODEL} (기본 llm=llama-3.1-8b-instruct 대신 사용)')

unified_memory = MemorySaver()

SYSTEM_PROMPT = (
    '당신은 4가지 핵심 역량을 갖춘 AI 에이전트입니다.\n\n'
    '역량:\n'
    '- Tool Use  : calculator, get_current_time, search_web 도구를 적극 활용\n'
    '- Memory    : 대화 맥락을 기억하여 일관된 응답\n'
    '- Planning  : 복잡한 요청을 단계별로 분해하여 처리\n'
    '- Reasoning : 논리적으로 추론하여 정확한 답변 제공\n\n'
    '규칙:\n'
    '- 항상 한국어로 답변하세요.\n'
    '- 도구가 필요한 경우 주저 없이 사용하세요.\n'
    '- 여러 단계가 필요한 경우 순서대로 진행하세요.'
)

unified_agent = create_agent(
    model=agent_llm,               # 기본 llm 이 아니라 도구 호출이 강한 agent_llm 사용
    tools=tool_list,
    checkpointer=unified_memory,
    system_prompt=SYSTEM_PROMPT,
)


def ask_agent(message: str, thread_id: str = 'unified') -> str:
    """통합 에이전트에 메시지를 보내고 도구 호출·결과를 출력한 뒤 답변을 반환한다."""
    config = {'configurable': {'thread_id': thread_id}}
    result = unified_agent.invoke(
        {'messages': [HumanMessage(content=message)]},
        config=config
    )
    from langchain_core.messages import AIMessage, ToolMessage as TM
    for msg in result['messages']:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  [도구] {tc['name']}({tc['args']})")
        elif isinstance(msg, TM):
            print(f'  [결과] {to_text(msg.content)[:80]}')
    return to_text(result['messages'][-1].content)  # 정규화 후 반환


# ── 복합 태스크 1: Tool Use + Reasoning ──────────────────────────────────
print('=' * 65)
print('복합 태스크 1: Tool Use + Reasoning')
print('질문: AI 트렌드 검색 + 현재 시각 + 3^8 계산')
print('=' * 65)
r1 = ask_agent(
    'AI 트렌드를 검색하고, 현재 시각도 알려준 다음, 3의 8제곱을 계산해줘.',
    thread_id='demo'
)
print(f'\n답변: {r1[:300]}')

# ── 복합 태스크 2: Memory 저장 ────────────────────────────────────────────
print('\n' + '=' * 65)
print('복합 태스크 2: 사용자 정보 저장 (Memory)')
print('=' * 65)
r2 = ask_agent(
    '제 이름은 이준혁이고 LangGraph 를 배우는 ML 엔지니어예요.',
    thread_id='demo'
)
print(f'\n답변: {r2[:150]}')

# ── 복합 태스크 3: Memory 활용 ───────────────────────────────────────────
print('\n' + '=' * 65)
print('복합 태스크 3: 이전 대화 기억 활용')
print('=' * 65)
r3 = ask_agent(
    '제 이름이 뭐고 무슨 일을 하고 있다고 했죠?',
    thread_id='demo'
)
print(f'\n답변: {r3[:150]}')

[에이전트 모델] qwen/qwen3-next-80b-a3b-instruct (기본 llm=llama-3.1-8b-instruct 대신 사용)
복합 태스크 1: Tool Use + Reasoning
질문: AI 트렌드 검색 + 현재 시각 + 3^8 계산


  [도구] search_web({'query': '최신 AI 트렌드 2024'})
  [도구] get_current_time({'dummy': ''})
  [도구] calculator({'expression': '3 ** 8'})
  [결과] '최신 AI 트렌드 2024' 웹 검색 결과 상위 5건:
1. [AI] 데이터분석, 인공지능 트렌드2024, LLM의 비상, 검색증강생성모델..
  [결과] 2026-07-25 15:58:58
  [결과] 3 ** 8 = 6561

답변: 2024년 최신 AI 트렌드는 다음과 같습니다:

1. **LLM(대형 언어 모델)의 비상**: 검색증강생성(RAG) 모델이 주목받으며, 정확한 정보 제공을 위한 실용적 AI 적용이 확대되고 있습니다.
2. **오픈소스 AI의 약진**: 대기업의 폐쇄적 플랫폼 대신, 오픈소스 모델이 개발자 사이에서 인기를 끌고 있습니다.
3. **멀티모달 AI**: 텍스트, 이미지, 음성 등을 통합해 이해하고 생성하는 AI 기술이 발전 중입니다.
4. **AI 에이전트**: 자율적으로 작업을 계획하고 실행하는 AI 에이전트가 마케팅, 고객 서비스 

복합 태스크 2: 사용자 정보 저장 (Memory)


  [도구] search_web({'query': '최신 AI 트렌드 2024'})
  [도구] get_current_time({'dummy': ''})
  [도구] calculator({'expression': '3 ** 8'})
  [결과] '최신 AI 트렌드 2024' 웹 검색 결과 상위 5건:
1. [AI] 데이터분석, 인공지능 트렌드2024, LLM의 비상, 검색증강생성모델..
  [결과] 2026-07-25 15:58:58
  [결과] 3 ** 8 = 6561

답변: 안녕하세요, 이준혁님! 😊  
LangGraph를 배우는 ML 엔지니어라고 하셨군요, 멋진 선택입니다! LangGraph는 LLM 기반의 복잡한 작업 흐름을 **그래프 기반으로 구성**할 수 있어, 에이전트 시스템, 다단계 추론, 상태 기반 워크플로우 구축에 매우 강력

복합 태스크 3: 이전 대화 기억 활용


  [도구] search_web({'query': '최신 AI 트렌드 2024'})
  [도구] get_current_time({'dummy': ''})
  [도구] calculator({'expression': '3 ** 8'})
  [결과] '최신 AI 트렌드 2024' 웹 검색 결과 상위 5건:
1. [AI] 데이터분석, 인공지능 트렌드2024, LLM의 비상, 검색증강생성모델..
  [결과] 2026-07-25 15:58:58
  [결과] 3 ** 8 = 6561

답변: 이전에 말씀하신 바에 따르면,  
- **이름**: **이준혁**님  
- **직업/역할**: **ML 엔지니어**이며, **LangGraph**를 배우고 계십니다.

맞습니까? 😊  
그렇다면, LangGraph 학습에 도움이 될 만한 예제나 개념 설명이 필요하신가요


---
## 정리

### 학습 요약

| 역량 | 핵심 API | 포인트 |
|------|----------|---------|
| **Tool Use** | `@tool`, `bind_tools()`, `ToolMessage` | LLM 이 도구 선택 → 실행 → 결과 해석 |
| **Memory** | `MemorySaver`, `thread_id` | 스레드별 독립 기억, 세션 간 지속 |
| **Planning** | `with_structured_output(PydanticSchema)` | 구조화된 계획 → 의존성 기반 실행 |
| **Reasoning** | CoT 프롬프트, `create_agent` | 단계별 추론으로 정확도 향상 |

### 참고 자료
- [LangGraph 공식 문서](https://langchain-ai.github.io/langgraph/)
- [LangChain Tools 가이드](https://python.langchain.com/docs/how_to/tool_calling/)
- [ReAct 논문](https://arxiv.org/abs/2210.03629)
- [Chain of Thought 논문](https://arxiv.org/abs/2201.11903)
